## Data Processing

In this notebook we will load and process all of the taxi data from 2011 to 2025.

The data is stored in data/raw as parquets.

It will be saved to data/processed as both parquets and csvs. 

We will save both the full dataset for each year, as well as the data for JFK and two time series for each year. One for a daily pickup taxi count at JFK and one for an hourly pickup taxi count at JFK.

We use our own process_taxi_data function that can be found in src/ This function will also perform some basic data cleaning as well.


In [1]:
from jfk_taxis import process_taxi_data, taxi_data_visuals, ts_plots, combine_ts, plot_full_ts, load_config
import pandas as pd

In [2]:
# Load config and project root
config, PROJECT_ROOT = load_config()

# Location of data
DATA_DIR_RAW = PROJECT_ROOT / config["data"]["data_path"] / config["data"]["raw_path"]

# Location to save reports
DATA_DIR_MAPS = PROJECT_ROOT / config["data"]["reports_path"] / config["data"]["maps_path"]

In [ ]:
# We do some inital data exploration on each year just to get a feel for the data, we only select columns "tpep_pickup_datetime" and "PULocationID" as otherwise we are trying to load approx 1.2GB of data for earlier years like 2011. 
# One of the interesting things in the plot is that for 2021 say we clearly have data that is not just from 2021, this will be cleaned up in process_taxi_data 
# which you can see inside src/jfk_taxis/data_processing.py

# Select the years to perfrom some basic EDA on, this can take a long time. Particularly with earlier years like 2011 as they have approx 170M rows which is a fair amount for pandas to handle
# so its recommend exploring only a few years. From 1_EDA.ipynb we know there were considerably more Yellow Taxi pick ups in earlier years hence they tend to have more rows and much larger file sizes as each row is one taxi trip.


# The other interesting thing is 2009 and 2010 are both using lat long data so have been excluded for now. 

# Years to perfrom some basic EDA on, the daily trip count is for the whole of NYC not just JFK airport

# Uncomment the following lines to run

years = [2011, 2015, 2020, 2025]

taxi_data_visuals(years)

Year:   0%|          | 0/3 [00:00<?, ?it/s]

Download files:   0%|          | 0/12 [00:00<?, ?it/s]

,tpep_pickup_datetime,PULocationID
0,2015-01-01 00:11:33,41
1,2015-01-01 00:18:24,166
2,2015-01-01 00:26:19,238
3,2015-01-01 00:45:26,162
4,2015-01-01 00:59:21,236


Shape: (146039231, 2)


,nulls
tpep_pickup_datetime,0
PULocationID,0


In [ ]:
# We provide a list of years of data to process
years = list(range(2011, 2026))

# We provide a list of features we want time series for
features = ["daily", "hour"]

# Process the data
process_taxi_data(years, features)

In [ ]:
# We combine the years into two single csvs:
combine_ts(years)

In [ ]:
# Plot the full daily time series
dir_path = "../data/processed/"
years = list(range(2011, 2026))

df_daily = pd.read_csv(f"{dir_path}ts_daily{years[0]}-{years[-1]}.csv")

plot_full_ts(df_daily, years)

Interestingly in the above you can seee the sharp dip and slow rise due to COVID. You can also see that yellow taxi numbers haven't yet recovered to their pre COVID numbers.

In [ ]:
# Plot the collected time series, we do an hourly plot for the whole year, hourly plot just for Janurary (the [x,y] determines which months to plot between in ts_plots) and a daily plot for the whole year

# You can see the effect of the COVID 2020 lockdowns in the data as well

years = [2011, 2016, 2020, 2025]

for year in years:
    file_hour = "ts_hour" + str(year) + ".csv"
    file_daily = "ts_daily" + str(year) + ".csv"
    
    df_hour = pd.read_csv(dir_path + file_hour)
    df_daily = pd.read_csv(dir_path + file_daily)

    ts_plots(df_hour, "hour", year, [])
    ts_plots(df_hour, "hour", year, [1, 1])
    ts_plots(df_daily, "daily", year, [])


What is potentially interesting about the above plots is there appears to be a periodic nature both within the daily and hourly plots of rapid up and down cycles of roughly similar length. This will be explored later in more detail but it's likely both a weekly cycle (so more demand say during the week than weekend) as well as daily demand, so more demand during the day than late at night potentially.